In [1]:
import os
import numpy as np
import pandas as pd

# =================================================================
# 1. SETUP & PATHS
# =================================================================
N_grid = 512

# Your target output directory
out_dir = "/home/storgaard/OneDrive/Speciale/C_256+2048"
os.makedirs(out_dir, exist_ok=True)

# TODO: Put the paths to the folders containing your .dat files here!
input_dir_2048 = r"/home/storgaard/OneDrive/concept/grendel_output/2048/" 
input_dir_256  = r"/home/storgaard/OneDrive/concept/grendel_output/256/"

# The exact models shown in your screenshot
# models = ["baseline", "dNeff_-0.5", "dNeff_-1", "dNeff_-1.5", "dNeff_-2", "dNeff_+0.5", "dNeff_+1", "dNeff_+1.5", "dNeff_+2", "fidm_-0.5", "fidm_-1", "fidm_-1.5", "fidm_-2", "fidm_+0.5", "fidm_+1", "fidm_+1.5", "fidm_+2", "test_1", "test_2", "test_3"] 
models = ['test_1']
redshifts = np.arange(0.00, 3.01, 0.1)  # z = 0.00, 0.01, ..., 3.00

# =================================================================
# 2. CALCULATE CUTOFFS
# =================================================================
k_nyq_2048 = np.pi / (2048 / N_grid)
k_nyq_256  = np.pi / (256 / N_grid)

# Transition point: Large box ends at 0.5 * k_nyq_2048
k_transition = 0.5 * k_nyq_2048

# End point: Small box ends at 0.8 * k_nyq_256
k_max_small = 0.75 * k_nyq_256

print(f"--- Stitching & Ratio Parameters ---")
print(f"Transition from 2048 to 256 at k = {k_transition:.3f}")
print(f"Final cutoff for 256 data at k   = {k_max_small:.3f}\n")

# =================================================================
# 3. HELPER FUNCTIONS
# =================================================================
def load_raw_pk(box_dir, model_name, z):
    """
    Loads raw P(k) files based on your CONCEPT output naming convention.
    """
    a = 1 / (1 + z)
    
    # Format 'a' to exactly 4 decimal places to perfectly match your files
    filename = f"powerspec/powerspec_size512_lpt2_a={a:.3f}"
    filepath = os.path.join(box_dir, model_name, filename)
    
    # usecols=[0, 1, 3] ensures we only grab k, modes and Pk, ignoring shot noise/other columns
    df = pd.read_csv(filepath, sep=r'\s+', comment='#', header=None, usecols=[0, 1, 3], names=['k', 'modes', 'Pk'])
    return df

def stitch_spectra(df_large, df_small):
    """
    Stitches the large and small box data at the calculated transition k.
    """
    # 1. Large box: k <= 0.5 * k_nyq_2048
    large_mask = df_large['k'] <= k_transition
    part_1 = df_large[large_mask]
    
    # 2. Small box: 0.5 * k_nyq_2048 < k <= 0.8 * k_nyq_256
    small_mask = (df_small['k'] <= k_max_small)
    part_2 = df_small[small_mask]
    
    # 3. Combine
    return pd.concat([part_1, part_2], ignore_index=True)

# =================================================================
# 4. EXECUTION LOOP: STITCH -> RATIO -> SAVE
# =================================================================
for z in redshifts:
    print(f"Processing z = {z:.2f}...")
    
    try:
        # Load and stitch the baseline
        ref_large = load_raw_pk(input_dir_2048, "planck", z)
        ref_small = load_raw_pk(input_dir_256, "planck", z)
        ref_stitched = stitch_spectra(ref_large, ref_small)
        
        for model in models:
            mod_large = load_raw_pk(input_dir_2048, model, z)
            mod_small = load_raw_pk(input_dir_256, model, z)
            mod_stitched = stitch_spectra(mod_large, mod_small)
            
            # THE FIX: Merge exactly on 'k' to bulletproof the alignment!
            merged = pd.merge(mod_stitched, ref_stitched, on='k', suffixes=('_mod', '_ref'))
            
            # Now divide the perfectly aligned arrays
            C_k = merged['Pk_mod'] / merged['Pk_ref']
            
            out_df = pd.DataFrame({
                'k': merged['k'],
                'Ck': C_k,
                'modes': merged['modes_mod'],  
                'Pk_DRMD': merged['Pk_mod'],  
                'Pk_Planck': merged['Pk_ref'],  
            })
            
            # Drop any lingering NaNs BEFORE saving to keep the text file pristine
            out_df = out_df.dropna()
            
            model_out_dir = os.path.join(out_dir, model)
            os.makedirs(model_out_dir, exist_ok=True)
            
            out_filename = f"Ck_z{z:.2f}.txt"
            out_path = os.path.join(model_out_dir, out_filename)
            
            out_df.to_csv(out_path, sep='\t', index=False, float_format='%.6e')
            print(f"  -> Saved {model}/{out_filename}")
            
    except FileNotFoundError as e:
        print(f"  [!] Missing data for z={z}: {e}")
        
print("\nSuccess! All stitched ratios have been saved.")

--- Stitching & Ratio Parameters ---
Transition from 2048 to 256 at k = 0.393
Final cutoff for 256 data at k   = 4.712

Processing z = 0.00...
  -> Saved test_1/Ck_z0.00.txt
Processing z = 0.10...
  -> Saved test_1/Ck_z0.10.txt
Processing z = 0.20...
  -> Saved test_1/Ck_z0.20.txt
Processing z = 0.30...
  -> Saved test_1/Ck_z0.30.txt
Processing z = 0.40...
  -> Saved test_1/Ck_z0.40.txt
Processing z = 0.50...
  -> Saved test_1/Ck_z0.50.txt
Processing z = 0.60...
  -> Saved test_1/Ck_z0.60.txt
Processing z = 0.70...
  -> Saved test_1/Ck_z0.70.txt
Processing z = 0.80...
  -> Saved test_1/Ck_z0.80.txt
Processing z = 0.90...
  -> Saved test_1/Ck_z0.90.txt
Processing z = 1.00...
  -> Saved test_1/Ck_z1.00.txt
Processing z = 1.10...
  -> Saved test_1/Ck_z1.10.txt
Processing z = 1.20...
  -> Saved test_1/Ck_z1.20.txt
Processing z = 1.30...
  -> Saved test_1/Ck_z1.30.txt
Processing z = 1.40...
  -> Saved test_1/Ck_z1.40.txt
Processing z = 1.50...
  -> Saved test_1/Ck_z1.50.txt
Processing z = 1